In [ ]:
# Ejecutar
#!pip install folium
#!pip install branca

In [2]:
from IPython.display import display, HTML

display(HTML(data="""
<style>
    div#notebook-container    { width: 95%; }
    div#menubar-container     { width: 65%; }
    div#maintoolbar-container { width: 99%; }a
</style>
"""))

# FOLIUM

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import chardet

import folium as fm
from folium import Marker, GeoJson
from folium.plugins import MarkerCluster, HeatMap, StripePattern

import geopandas as gpd
from geopandas import GeoSeries
from shapely.geometry import Point, LineString

import branca as br 

- <a href='#1.'>I. ¿Qué es Folium?</a>
     - <a href='#1.1.'>1.1. Importando datos de Minería</a>
     - <a href='#1.2.'>1.2. Creando mapas: Marcadores</a>
     - <a href='#1.3.'>1.3. Marcador Ballon</a>

# <a id ='1.'>I. ¿Qué es Folium?</a>

* `Folium` es una potente librería de Python que te ayuda a crear diversos tipos de mapas Leaflet. Por defecto, Folium crea un mapa en un archivo HTML independiente.

* **Ventajas:**
    * Facilita la visualización de datos que han sido manipulados en Python sobre un mapa interactivo de Leaflet. Permite tanto la vinculación de datos a un mapa para crear mapas de coropletas, como la integración de visualizaciones enriquecidas en formato vectorial, ráster o HTML como marcadores en el mapa.

    * La librería cuenta con una serie de mapas base (tilesets) integrados de OpenStreetMap, Mapbox y Stamen, y admite mapas base personalizados mediante claves API de Mapbox o Cloudmade. Folium soporta superposiciones de imágenes, video, GeoJSON y TopoJSON.

* **Punto de partida:**
    * `Folium.Map(location = [LAT0,LON0], zoom_start = 12)`
    * `location` contiene unas coordenadas específicas y `zoom` muestra la distancia desde dicha coordenada.

## <a id ='1.1.'>1.1. Importando datos de Minería</a>

In [11]:
#Obteniendo el formato del archivo

base = open(r'data/MINING.csv', 'rb').read()
det = chardet.detect(base)
charenc = det['encoding']
charenc

'UTF-8-SIG'

Las columnas nuevas para el análisis son:
* Método de explotación.
* Titular: Propietaria de la explotación.
* Unidad: Lugar donde se hace la explotación.
* Producto: Mineral objetivo de la explotación.

In [12]:
# Información geográfica de Mining:

MINING = pd.read_csv( r'data/MINING.csv', encoding = charenc)
MINING

,UBIGEO,MÉTODO DE EXPLOTACIÓN,TITULAR,UNIDAD,REGION,PROVINCIA,DISTRITO,PRODUCTO,LONGITUD,LATITUD
0,20201,MINERÍA SUBTERRÁNEA,COMPAÑIA MINERA LINCUNA S.A,HUANCAPETI,ANCASH,AIJA,AIJA,"As, Bi, Mn, Pb, Zn, Au, Ag",-77.531000,-9.753000
1,220902,MINERíA NO METÁLICA,QUIMPAC S.A.,SALINAS PILLUANA,SAN MARTIN,SAN MARTIN,ALBERTO LEVEAU,Sal,-76.267000,-6.742000
2,200502,MINERíA NO METÁLICA,COMPAÑIA MINERA AGREGADOS CALCAREOS S.A.,CERRO BLANCO,PIURA,PAITA,AMOTAPE,Bentonita,-81.027000,-4.836000
3,210802,MINERÍA SUBTERRÁNEA,MINSUR S.A.,QUENAMARI-SAN RAFAEL,PUNO,MELGAR,ANTAUTA,Sn,-70.491600,-14.133500
4,150502,MINERíA NO METÁLICA,COMPAÑIA MINERA LAS CAMELIAS S.A.,PROMESA 345,LIMA,CAÑETE,ASIA,Arcillas,-76.530467,-12.736480
...,...,...,...,...,...,...,...,...,...,...
137,120810,MINERÍA SUBTERRÁNEA,COMPAÑIA MINERA ARGENTUM S.A.,MANUELITA,JUNIN,YAULI,YAULI,"Pb, Zn, Ag, Cu",-76.096077,-11.628891
138,120810,MINERÍA SUBTERRÁNEA,VOLCAN COMPAÑÍA MINERA S.A.A.,SAN CRISTOBAL,JUNIN,YAULI,YAULI,"Cu, Pb, Zn, Ag",-75.484539,-12.018773
139,120810,MINERíA NO METÁLICA,MINERA CHINALCO PERU S.A.,TUNSHURUCO,JUNIN,YAULI,YAULI,Caliza,-76.139792,-11.656098
140,60508,MINERíA NO METÁLICA,CEMENTOS PACASMAYO S.A.A.,TEMBLADERA,CAJAMARCA,CONTUMAZA,YONAN,Caliza,-79.124000,-7.247000


In [13]:
# Actividad minerea en el distrito de Yauli
MINING_YAULI = MINING[MINING.DISTRITO == "YAULI"]
MINING_YAULI

,UBIGEO,MÉTODO DE EXPLOTACIÓN,TITULAR,UNIDAD,REGION,PROVINCIA,DISTRITO,PRODUCTO,LONGITUD,LATITUD
134,120810,MINERÍA SUBTERRÁNEA,COMPAÑIA MINERA CASAPALCA S.A.,AMERICANA,JUNIN,YAULI,YAULI,"Zn, Ag, Cu, Pb",-76.199576,-11.698612
135,120810,MINERÍA SUBTERRÁNEA,COMPAÑIA MINERA ARGENTUM S.A.,ANTICONA,JUNIN,YAULI,YAULI,"Zn, Ag, Cu, Pb",-76.171596,-11.637246
136,120810,MINERÍA SUBTERRÁNEA,VOLCAN COMPAÑÍA MINERA S.A.A.,CARAHUACRA,JUNIN,YAULI,YAULI,"Zn, Ag, Cu, Pb",-76.070389,-11.740413
137,120810,MINERÍA SUBTERRÁNEA,COMPAÑIA MINERA ARGENTUM S.A.,MANUELITA,JUNIN,YAULI,YAULI,"Pb, Zn, Ag, Cu",-76.096077,-11.628891
138,120810,MINERÍA SUBTERRÁNEA,VOLCAN COMPAÑÍA MINERA S.A.A.,SAN CRISTOBAL,JUNIN,YAULI,YAULI,"Cu, Pb, Zn, Ag",-75.484539,-12.018773
139,120810,MINERíA NO METÁLICA,MINERA CHINALCO PERU S.A.,TUNSHURUCO,JUNIN,YAULI,YAULI,Caliza,-76.139792,-11.656098


* Siempre para plotear el mapa, necesitamos el Centroide
* **Centroide:** Promedio de las latitudes (x) y longitudes (y).

## <a id ='1.2.'>1.2. Mapa base</a>

* Creamos un mapa base de referencia.
* En este caso, el distrito de Yauli.

In [18]:
zoom_start = 12
lat_mining = MINING_YAULI["LATITUD"].mean()
long_mining = MINING_YAULI["LONGITUD"].mean()

a = fm.Map( location = [lat_mining, long_mining], tiles="OpenStreetMap", zoom_start = 11, control_scale=True)
a

Si no carga el mapa: 
* Presionar `Ctrl + Shift + P` en el teclado.
* Escribir *Workspace Trust* y elegir *Workspaces: Manage Workspace Trust*.
* Colocar el folder de trabajo como entorno confiable.
* Hacer click en *Clear All Outputs*.
* Cerrar VS Code y volverlo a abrir.

## <a id ='1.3.'>1.3. Marcadores clásicos</a>

**A. MARCADOR PREDETERMINADO:**

In [15]:
# fm: Folium
# location: Punto central en donde se ploteará el mapa.
# tiles: Tipo de mapa a ver.
# zoom_start= El zoom que queremos.
# control_scale=True: Para que aparezca la escala en el mapa.

#tooltip: Aparece cuando pasamos el mouse por encima del marcador.
tooltip = "Click me!" 

# Marcadores que pongo encima del mapa creado antes:
fm.Marker(    [-11.637246, -76.171596], 
                  popup="<i>COMPAÑIA MINERA CASAPALCA S.A.</i>", 
                  tooltip=tooltip
                   ).add_to(a)

fm.Marker(    [-11.628891, -76.096077 ], 
                  popup="<b><i>COMPAÑIA MINERA ARGENTUM S.A.</i></b>", 
                  tooltip=tooltip
                 ).add_to(a)

a

**B. MARCADOR INFO-SIGN:**

* **GITHUB PARA MÁS MARCADORES:** [LINK](https://github.com/lennardv2/Leaflet.awesome-markers) 

* Iteración para agregar Marcadores a todo el DataFrame de Minería en Yauli:
* **OBJETIVO:** Agregar marcadores a todos los puntos de una tabla. En ese caso, la minería en Yauli al mapa creado antes.

In [16]:
for index, row in MINING_YAULI.iterrows():
    
    fm.Marker([row['LATITUD'], row['LONGITUD']], popup= row['TITULAR'],
          radius = 50,
          icon=fm.Icon(color="red", icon="info-sign"),
          color="#3186cc",  
          fill=True,
          fill_color="#3186cc",
          tooltip=tooltip).add_to(a)
a

## <a id ='1.4.'>1.4. Marcadores eficientes: Circulares (circle market)</a>

* Si tenemos que graficar cientos o miles de puntos, usemos este mercador.
* Si intentamos con los anteriores, nuestra PC no podrá graficarlos bien (o no los cargará) porque no son eficientes.

* Para ver solo los círculos azules, ejecutar el mapa base del punto 1.2 y luego volver aquí.

In [19]:
tooltip = "Click"

# Lista de tipos de minerales que se encuentran en Yauli.

#mineral = ["CASAPALCA (Zinc)","ARGENTUM (Cobre)","VOLCAN (Mercurio)", "ARGENTUM (Plomo)", "VOLCAN (Plata)", "CHINALCO (Caliza)"]

# Adding circle and mark. 
# popup save information 
# CircleMarker uses Pixels
for _, row in MINING_YAULI.iterrows():
    
    fm.Circle([row['LATITUD'], row['LONGITUD']], popup=row['TITULAR'], # Llamamos a folium.Circle y le pasamos la latitud y longitud de cada fila del DataFrame
              radius = 10000, # Está medido en metros, por lo que 10000 equivale a 10 km de radio
              fill=True,
              fill_color="#3186cc",
              tooltip=tooltip).add_to(a)
a

# <a id ='2.'>II. Tiles</a>

* Son las diferentes formas de visualizar el mapa.
* Varía según el estilo de la empresa que haga cada visualización.
* Tipos de Tiles: 
    * [Github](https://python-visualization.github.io/folium/latest/user_guide/raster_layers/tiles.html)
    * [Leaflet](https://leaflet-extras.github.io/leaflet-providers/preview/)

In [22]:
zoom_start = 10
lat_mining = MINING_YAULI["LATITUD"].mean()
long_mining = MINING_YAULI["LONGITUD"].mean()

# "Cartodb Positron"
a = fm.Map(location = [lat_mining, long_mining], tiles="Cartodb Positron", zoom_start = zoom_start, control_scale=True)

# Marcadores:
for index, row in MINING_YAULI.iterrows():
    fm.Marker([row['LATITUD'], row['LONGITUD']], popup= row['TITULAR'],
          radius = 50,
          icon=fm.Icon(color="red", icon="info-sign"),
          color="#3186cc",  
          fill=True,
          fill_color="#3186cc",
          tooltip=tooltip).add_to(a)

# Circles
for _, row in MINING_YAULI.iterrows():
    
    fm.Circle([row['LATITUD'], row['LONGITUD']], popup=row['TITULAR'],
              radius = 10000,
              fill=True,
              fill_color="#3186cc",
              tooltip=tooltip).add_to(a)

a 

# <a id ='3.'>III. Archivos de Peru GeoJson</a>

* Es el formato de datos al cual están migrando desde Shapefile.
* Motivo: No son 5 o 6 archivos por separado, es solo un archivo con toda la informacaión geográfica.

## <a id ='3.1.'>3.1. Nivel distrital</a>

In [ ]:
# Notamos que usamos Geopandas para leer GeoJson. 
# Debemos subir el archivo en formato GeoJson para poder usar el paquete Choropleth.

distritos = gpd.read_file(r'data/geojson_files/peru_distrital_simple.geojson')
distritos

,OBJECTID,IDDIST,IDDPTO,IDPROV,NOMBDIST,NOMBPROV,NOMBDEP,DCTO,LEY,FECHA,NOM_CAP,SHAPE_LENG,SHAPE_AREA,SHAPE_LE_1,SHAPE_AR_1,AREA_MINAM,geometry
0,1,230110,23,2301,CORONEL GREGORIO ALBARRACIN LANCHIPA,TACNA,TACNA,LEY,27415,02/02/2001,ALFONSO UGARTE,0.570510,0.016140,0.570195,0.015990,18834.14,"POLYGON ((-70.14409 -18.09309, -70.17512 -18.1..."
1,2,230108,23,2301,POCOLLAY,TACNA,TACNA,LEY,13069,15/01/1959,POCOLLAY,0.883871,0.022816,0.897169,0.022961,27073.52,"POLYGON ((-69.90467 -17.95829, -69.98287 -18.0..."
2,3,230103,23,2301,CALANA,TACNA,TACNA,LEY,S/N,20/08/1872,CALANA,0.446736,0.009458,0.445963,0.009383,11063.99,"POLYGON ((-70.09201 -17.98026, -70.17243 -18.0..."
3,4,230101,23,2301,TACNA,TACNA,TACNA,-,-,EPOCA INDEP.,TACNA,2.758951,0.209156,2.758123,0.209177,246365.27,"POLYGON ((-70.235 -17.99231, -70.2371 -18.0224..."
4,5,230109,23,2301,SAMA,TACNA,TACNA,-,-,EPOCA INDEP.,LAS YARAS,1.515506,0.096789,1.513660,0.096766,113953.51,"POLYGON ((-70.42374 -17.88983, -70.51323 -17.9..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1829,1830,160303,16,1603,TIGRE,LORETO,LORETO,LEY,9815,02/07/1943,INTUTU,9.211057,1.637168,9.179725,1.637432,2011378.30,"POLYGON ((-74.96515 -2.36565, -74.94959 -2.458..."
1830,1831,160107,16,1601,NAPO,MAYNAS,LORETO,LEY,9815,02/07/1943,SANTA CLOTILDE,11.380550,1.986357,11.359436,1.985362,2440805.01,"POLYGON ((-72.9379 -2.83906, -72.97311 -2.9301..."
1831,1832,160109,16,1601,PUTUMAYO,MAYNAS,LORETO,LEY,9815,02/07/1943,SAN ANTONIO DEL ESTRECHO,16.256407,2.884865,16.232424,2.884722,3555516.31,"POLYGON ((-73.53075 -1.45793, -73.47857 -1.522..."
1832,1833,160110,16,1601,TORRES CAUSANA,MAYNAS,LORETO,LEY,9815,02/07/1943,PANTOJA,6.592491,0.609698,6.572157,0.609290,749185.08,"POLYGON ((-74.8737 -0.95012, -74.88779 -1.0063..."


In [ ]:
# Nos quedamos con las columnas IDDIST y geometry, que son las que necesitamos para hacer el merge con el DataFrame MINING_YAULI.
# Renombramos la columna IDDIST a UBIGEO1 y convertimos a int64 para poder hacer el merge
distritos1 = distritos[['IDDIST', 'geometry']]
distritos1 = distritos1 .rename({'IDDIST':'UBIGEO1'}, axis =1 )
distritos1['UBIGEO1'] = distritos1['UBIGEO1'].astype(str).astype(np.int64)
distritos1

,UBIGEO1,geometry
0,230110,"POLYGON ((-70.14409 -18.09309, -70.17512 -18.1..."
1,230108,"POLYGON ((-69.90467 -17.95829, -69.98287 -18.0..."
2,230103,"POLYGON ((-70.09201 -17.98026, -70.17243 -18.0..."
3,230101,"POLYGON ((-70.235 -17.99231, -70.2371 -18.0224..."
4,230109,"POLYGON ((-70.42374 -17.88983, -70.51323 -17.9..."
...,...,...
1829,160303,"POLYGON ((-74.96515 -2.36565, -74.94959 -2.458..."
1830,160107,"POLYGON ((-72.9379 -2.83906, -72.97311 -2.9301..."
1831,160109,"POLYGON ((-73.53075 -1.45793, -73.47857 -1.522..."
1832,160110,"POLYGON ((-74.8737 -0.95012, -74.88779 -1.0063..."


## <a id ='3.2.'>3.2. Nivel departamental</a>

In [25]:
# Hago lo mismo pero con departamentos
# Me quedo con las columnas FIRST_IDDP y geometry, que son las que necesitamos para hacer el merge con el DataFrame MINING_YAULI.
# Renombro a la columna FIRST_IDDP como UBIGEO2 y le agrego "0000" al final

dpto = gpd.read_file(r'data/geojson_files/peru_departamental_simple.geojson')
dpto1 = dpto[['FIRST_IDDP', 'geometry']]
dpto1 = dpto1.rename({'FIRST_IDDP':'UBIGEO2'}, axis =1 )
dpto1['UBIGEO2'] = dpto1['UBIGEO2'] + "0000"
dpto1['UBIGEO2'] = dpto1['UBIGEO2'].astype(str).astype(np.int64)
dpto1

,UBIGEO2,geometry
0,10000,"POLYGON ((-77.75893 -6.96451, -77.84586 -6.976..."
1,20000,"POLYGON ((-77.31749 -8.53015, -77.28903 -8.589..."
2,30000,"POLYGON ((-72.47177 -14.6614, -72.57725 -14.68..."
3,40000,"POLYGON ((-75.07333 -15.44294, -75.04965 -15.4..."
4,50000,"POLYGON ((-74.34595 -12.17374, -74.32187 -12.2..."
5,60000,"POLYGON ((-79.32259 -7.02568, -79.29663 -6.999..."
6,70000,"POLYGON ((-77.1871 -11.82836, -77.12605 -11.82..."
7,80000,"POLYGON ((-72.47177 -14.6614, -72.4617 -14.605..."
8,90000,"POLYGON ((-75.05905 -14.12962, -75.10884 -14.0..."
9,100000,"POLYGON ((-77.31749 -8.53015, -77.26408 -8.467..."


### Data to plot- IDH, GDP , Poverty Rate 

In [ ]:
base = open(r'data/Poverty.csv', 'rb').read()
det = chardet.detect(base)
charenc = det['encoding']

poverty = pd.read_csv( r'data/Poverty.csv', encoding = charenc)
poverty

,UBIGEO1,UBIGEO2,DEPARTAMENTO,PROVINCIA,DISTRITO,POVERTY_RATE,PBI_PC,IDH
0,10101,10000,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,9.034625,3449.564733,41.8
1,10102,10000,AMAZONAS,CHACHAPOYAS,ASUNCIÓN,36.519949,3449.564733,41.8
2,10103,10000,AMAZONAS,CHACHAPOYAS,BALSAS,45.732962,3449.564733,41.8
3,10104,10000,AMAZONAS,CHACHAPOYAS,CHETO,39.169782,3449.564733,41.8
4,10105,10000,AMAZONAS,CHACHAPOYAS,CHILIQUÍN,53.045662,3449.564733,41.8
...,...,...,...,...,...,...,...,...
1868,250302,250000,UCAYALI,PADRE ABAD,IRAZOLA,12.899560,4059.106491,48.4
1869,250303,250000,UCAYALI,PADRE ABAD,CURIMANÁ,9.014545,4059.106491,48.4
1870,250304,250000,UCAYALI,PADRE ABAD,NESHUYA,11.946857,4059.106491,48.4
1871,250305,250000,UCAYALI,PADRE ABAD,ALEXANDER VON HUMBOLDT,12.899560,4059.106491,48.4


* El IDH y el PBI per cápita tienen el mismo valor aparente porque solo tienen a nivel departamental como máximo.
* El Poverty_rate (ratio de promesa) sí está a nivel distrital.

In [ ]:
#Check variables´ types to make math operations, merge datasets, etc
poverty.dtypes

In [ ]:
# From district level to administrative region

poverty2 = poverty.drop_duplicates(subset=['UBIGEO2'])
# Drop callao
poverty2 = poverty2.drop(689)
poverty2['IDH'] = poverty2['IDH'].astype(str).astype(float)

In [ ]:
poverty2['IDH'] = poverty2['IDH'].astype(str).astype(float)
poverty2['PBI_PC'] = poverty2['PBI_PC'].astype(str).astype(float)
poverty2

## 2. Choropleth 

In [ ]:
poverty

In [ ]:
# government palace coordinates

lat_palacio = -12.0757538
long_palacio = -76.9863174
zoom_start = 5

z = fm.Map(location = [lat_palacio, long_palacio], tiles='cartodbpositron', zoom_start = zoom_start)

# Mandatory: geo_data in GeoJson format
# columns: variables from economics indicators data set
# Atention !!! key_on: commom variable between geodata and data "feature.properties.(name of variable)"


fm.Choropleth(
    geo_data=distritos1,
    data=poverty,
    columns=['UBIGEO1', 'POVERTY_RATE'],
    key_on="feature.properties.UBIGEO1",
    fill_color="YlOrRd",
    fill_opacity=0.8,
    line_opacity=0.2,
    legend_name="Poverty Rate (%)",
    smooth_factor=0,
    Highlight= True,
    line_color = "#0000",
    overlay=True,
    nan_fill_color = "White"  # fill white missing values 
    ).add_to(z)

#fm.LayerControl().add_to(z)

# Save in a html format 

#z.save("Poverty_Map.html")

z

In [ ]:
# Add quantile on legend 
bins = list(poverty2["IDH"].quantile([0, 0.2, 0.4, 0.6,0.8, 1]))
bins

In [ ]:
lat_palacio = -12.0757538
long_palacio = -76.9863174

z = fm.Map(location = [lat_palacio, long_palacio], zoom_start = 5)



fm.Choropleth(geo_data=dpto1,
            data=poverty2,
            columns=["UBIGEO2", "IDH"],
            key_on="feature.properties.UBIGEO2",
            fill_color="Reds",  
            fill_opacity=0.8,
            legend_name="Human development index",
            bins = bins,
            reset = True
            ).add_to(z)

fm.LayerControl().add_to(z)

z

z.save("IDH_map_peru.html")

In [ ]:
lat_palacio = -12.0757538
long_palacio = -76.9863174


#add quantils

bins = list(poverty2["IDH"].quantile([0, 0.2, 0.4, 0.6,0.8, 1]))

z = fm.Map(location = [lat_palacio, long_palacio], zoom_start = 5)

fm.Choropleth(geo_data=dpto1,
            data=poverty2,
            columns=["UBIGEO2", "IDH"],
            key_on="feature.properties.UBIGEO2",
            fill_color="Reds",  
            fill_opacity=0.8,
            legend_name="Human development index",
            bins = bins,
            reset = True
            ).add_to(z)

#Merge both dataset to add informaction by region 
data_both = pd.merge(dpto1, poverty2, how="inner", on="UBIGEO2")
data_both = data_both.to_json()    # from pandas to Json 

# Color and opacity of each region 
style_function = lambda x: {'fillColor': '#ffffff', 
                            'color':'#000000', 
                            'fillOpacity': 0.1, 
                            'weight': 0.1}

# Color and opacity of each selected region 

highlight_function = lambda x: {'fillColor': '#000000', 
                                'color':'#000000', 
                                'fillOpacity': 0.5, 
                                'weight': 0.1}

details = fm.features.GeoJson(
    data = data_both,
    style_function=style_function, 
    control=False,
    highlight_function=highlight_function, 
    tooltip=fm.features.GeoJsonTooltip(
        fields=['DEPARTAMENTO','IDH'],   #Variable selection
        aliases=['Administrative Region', 'Human Development Index'],  # renames
        style=("background-color: white; color: #333333; font-family: arial; font-size: 12px; padding: 10px;") 
    )
)

# Add new features 

z.add_child(details)
z.keep_in_front(details)

z

## 3. Heat Map and Interactive Map

In [ ]:
#Gettting the character format

base = open(r'../_data/Folium/enaho.csv', 'rb').read()
det = chardet.detect(base)
charenc = det['encoding']

ENAHO = pd.read_csv( r'../_data/Folium/enaho.csv', encoding = charenc)
ENAHO

In [ ]:
# beneficiary households of the universal bond 
bono_uni= ENAHO[ENAHO.bono_uni == 1]
bono_uni

In [ ]:
# Coordinamtes in a list 
hogares = list(zip(bono_uni['latitud'], bono_uni['longitud']))
hogares

In [ ]:
lat_palacio = -12.0757538
long_palacio = -76.9863174

# List of tiles 
z = fm.Map(location = [lat_palacio, long_palacio], zoom_start = 12)

# Cluster Map
MarkerCluster( hogares, name = 'Cluster' ).add_to(z)
z

In [ ]:
# Heat Map 
HeatMap(data=bono_uni[['latitud', 'longitud']], radius=20, name = 'Heatmap').add_to(z)
z

In [ ]:
#Add different kind of tiles 
Tiles = ["stamenterrain","stamenwatercolor","cartodbpositron","openstreetmap", "cartodbdark_matter"]

for i in Tiles:
    fm.TileLayer(i, name = i, control = True).add_to(z)
    
fm.LayerControl(collapsed=False).add_to(z)

z.save("cluster_heatmap_bono.html")
z


In [ ]:
# beneficiary households of the universal bond 
bono_uni= ENAHO[ENAHO.bono_uni == 1]

# Coordinamtes in a list 

hogares = list(zip(bono_uni['latitud'], bono_uni['longitud']))

# List of tiles 

Tiles = ["stamenterrain","stamenwatercolor","cartodbpositron","openstreetmap", "cartodbdark_matter"]

z = fm.Map(location = [lat_palacio, long_palacio], zoom_start = 12)


# Heat Map 
HeatMap(data=bono_uni[['latitud', 'longitud']], radius=20, name = 'Heatmap').add_to(z)

# Cluster Map
MarkerCluster(hogares, name = 'Cluster').add_to(z)

#Add different kind of tiles 

for i in Tiles:
    fm.TileLayer(i, name = i, control = True).add_to(z)
    
fm.LayerControl(collapsed=False).add_to(z)
z.save("cluster_heatmap_bono.html")

## 4. Marker

In [ ]:
bonogas = ENAHO[ENAHO.bonogas == 1]

bn = fm.Map(location = [lat_palacio, long_palacio], zoom_start = 12, control_scale=True)

# Loop for rows

for idx, row in bonogas.iterrows():
    Marker([row['latitud'], row['longitud']]).add_to(bn)

bn

## 5. Circles

In [ ]:
base1 = ENAHO[ENAHO['nbi1'] == 1]
base2 = ENAHO[ENAHO['nbi2'] == 1]
base3 = ENAHO[ENAHO['nbi3'] == 1]
base4 = ENAHO[ENAHO['nbi4'] == 1]
base5 = ENAHO[ENAHO['nbi5'] == 1]

In [ ]:
globals()["base1"]

In [ ]:
#unsatisfied basic need

#nbi1: Inadequate housing
#nbi2: Overcrowded housing
#nbi3: Housing without SSHH
#nbi4: School non-attendance
#nbi5: High economic dependency

base1 = ENAHO[ENAHO['nbi1'] == 1]
base2 = ENAHO[ENAHO['nbi2'] == 1]
base3 = ENAHO[ENAHO['nbi3'] == 1]
base4 = ENAHO[ENAHO['nbi4'] == 1]
base5 = ENAHO[ENAHO['nbi5'] == 1]

m = fm.Map(location = [lat_palacio, long_palacio], zoom_start = 12.5, control_scale=True)

colors = ['forestgreen','darkred', 'blue', 'lime']

for j in range(1,5):
        
    for idx, row in globals()[f'base{j}'].iterrows():
        fm.Circle([row['latitud'], row['longitud']], radius = 200, color = colors[j-1]).add_to(m)

m

In [ ]:
# Marks and unsatisfied basic need


m = fm.Map(location = [lat_palacio, long_palacio], zoom_start = 11, control_scale=True)

colors = ['purple','lightgreen', 'red', 'darkblue', 'orange']
nbi = ['Vivienda inadecuada','Vivienda con hacinamiento', 'Vivienda sin SS.HH', 'Inasistencia escolar', 'Alta depenencia Económica']

for j in range(1,5):
        
    for idx, row in globals()[f'base{j}'].iterrows():
         Marker([row['latitud'], row['longitud']], icon=fm.Icon(color=colors[j-1]), popup= nbi[j-1]).add_to(m)

        
m    

## 6. Interactive Map part II

In [ ]:
# Solidaridad Hospital Centers 

base = open(r'../_data/Folium/Solidaridad_Center.csv', 'rb').read()
det = chardet.detect(base)
charenc = det['encoding']

h_solidaridad = pd.read_csv( r'../_data/Folium/Solidaridad_Center.csv', encoding = charenc)
h_solidaridad

In [ ]:
# Function create table by each Health center using html. This funtion will be aplly by each row
# Almost alway each code on html requires a beginnig <p> and ending </p> 

def visual_html(i):
 
    # information by Health center 

    
    district = h_solidaridad['distrito'].iloc[i]                             
    direction = h_solidaridad['direction'].iloc[i]                           
    atencion = h_solidaridad['Schedule'].iloc[i]  
    phone = h_solidaridad['phone'].iloc[i]  
    espec = h_solidaridad['especialidades'].iloc[i]
    beds = h_solidaridad['available_beds'].iloc[i]
    covid = h_solidaridad['Prueba_Covid'].iloc[i]
    vacunation = h_solidaridad['Centro_vacunacion'].iloc[i]
    Health_center = h_solidaridad['Health_center'].iloc[i]
    
    # Color by each column of table 
    
    left_col_colour = "#FA8072"
    right_col_colour = "#BDC3C7"
    
    html = """<!DOCTYPE html>
<html>

<head>
    <p> Solidaridad Health Center </p>

</head>
    <table style="height: 126px; width: 350px;">  <!-- Comment: Create a teable. -->

<!-- Add information  -->

<tbody> 
<tr>

<!-- Add color by column -->

<td style="background-color: """+ left_col_colour +""";"><span style="color: #ffffff;">District of Lima</span></td>
<td style="width: 150px;background-color: """+ right_col_colour +""";">{}</td>""".format(district) + """
</tr>
<tr>
<td style="background-color: """+ left_col_colour +""";"><span style="color: #ffffff;">Direction</span></td>
<td style="width: 150px;background-color: """+ right_col_colour +""";">{}</td>""".format(direction) + """
</tr>
<tr>
<td style="background-color: """+ left_col_colour +""";"><span style="color: #ffffff;">Openning Hour</span></td>
<td style="width: 150px;background-color: """+ right_col_colour +""";">{}</td>""".format(atencion) + """
</tr>
<tr>
<td style="background-color: """+ left_col_colour +""";"><span style="color: #ffffff;">Phone - number</span></td>
<td style="width: 150px;background-color: """+ right_col_colour +""";">{}</td>""".format(phone) + """
</tr>
<tr>
<td style="background-color: """+ left_col_colour +""";"><span style="color: #ffffff;">Number of medical specialties</span></td>
<td style="width: 150px;background-color: """+ right_col_colour +""";">{}</td>""".format(espec) + """
</tr>
<tr>
<td style="background-color: """+ left_col_colour +""";"><span style="color: #ffffff;">Available beds</span></td>
<td style="width: 150px;background-color: """+ right_col_colour +""";">{}</td>""".format(beds) + """
</tr>
<tr>
<td style="background-color: """+ left_col_colour +""";"><span style="color: #ffffff;">Covid test</span></td>
<td style="width: 150px;background-color: """+ right_col_colour +""";">{}</td>""".format(covid) + """
</tr>
<tr>
<td style="background-color: """+ left_col_colour +""";"><span style="color: #ffffff;">Vaccination center</span></td>
<td style="width: 150px;background-color: """+ right_col_colour +""";">{}</td>""".format(vacunation) + """
</tr>

</tbody>
</table>
</html>
"""
    return html

In [ ]:
ubication = h_solidaridad['latitud'].mean(), h_solidaridad['longitud'].mean()  # Average point

sol = fm.Map(location = ubication, zoom_start=12)

for i in range(0,len(h_solidaridad)):
    html = visual_html(i)

    iframe = br.element.IFrame(html=html,width=350,height=300)
    popup = fm.Popup(iframe, parse_html=True)
    
    fm.Marker([h_solidaridad['latitud'].iloc[i],h_solidaridad['longitud'].iloc[i]],
                  popup=popup, icon=fm.Icon(color= 'blue', icon='medkit', prefix="fa")).add_to(sol)

sol.save("hospital_solidaridad.html")
sol

In [ ]:
# Alternative

def visual_html(i):
    
        html="""
        <h4>Direction: </h4>""" + str(h_solidaridad.iloc[i]['distrito']) + " - " + str(h_solidaridad.iloc[i]['direction']) +\
         """<h4>Phone - number:</h4>""" + str(h_solidaridad.iloc[i]['phone']) +\
        """<h4>Openning hour:</h4>""" + str(h_solidaridad.iloc[i]['Schedule']) +\
         """<h4>Number of medical specialties:</h4>""" + str(h_solidaridad.iloc[i]['especialidades']) +\
         """<h4>Available_beds:</h4>""" + str(h_solidaridad.iloc[i]['available_beds']) +\
        """<h4>Covid Test:</h4>""" + str(h_solidaridad.iloc[i]['Prueba_Covid']) +\
        """<h4>Vaccination center:</h4>""" + str(h_solidaridad.iloc[i]['Centro_vacunacion']) 
        return html


In [ ]:
ubication = h_solidaridad['latitud'].mean(), h_solidaridad['longitud'].mean()

sol = fm.Map(location = ubication, zoom_start=12)

for i in range(0,len(h_solidaridad)):
    html = visual_html(i)

    iframe = br.element.IFrame(html=html,width=350,height=300)
    popup = fm.Popup(iframe,parse_html=True)
    
    fm.Marker([h_solidaridad['latitud'].iloc[i],h_solidaridad['longitud'].iloc[i]],
                  popup=popup,icon=fm.Icon(color= 'red', icon='medkit', prefix="fa")).add_to(sol)

sol.save("hospitals.html")

sol

### References:

##### MINEM Geografic mining centers 

http://www.minem.gob.pe/_publicaSector.php?idSector=1&idCategoria=24

##### Poverty map at distric level

https://www.inei.gob.pe/cifras-de-pobreza/

##### Geo-spatial information

https://visor.geoperu.gob.pe/

##### Folium 

https://python-visualization.github.io/folium/index.html

https://www.kaggle.com/alexisbcook/interactive-maps

https://www.kaggle.com/dabaker/fancy-folium

https://towardsdatascience.com/how-to-step-up-your-folium-choropleth-map-skills-17cf6de7c6fe

https://www.kaggle.com/mbnb8317/ds4c-tutorial-all-about-folium-pydeck